<a href="https://colab.research.google.com/github/cermegno/langgraph-learning/blob/main/05-nvidia-multi-step-researcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-step Researcher


In [ ]:
!pip install langgraph langchain langchain-nvidia-ai-endpoints langchain-text-splitters langchain-core langchain-community --quiet

In [ ]:
import os
from typing import List, Annotated, TypedDict, Literal, Union
from langgraph.graph import StateGraph, START, END
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage, AIMessage
import json

## Nvidia setup

In [ ]:
# Initialize the LLM
from google.colab import userdata
userdata.get('apikey')
os.environ["NVIDIA_API_KEY"] = apikey
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

In [ ]:
class State(TypedDict):
    user_goal: str
    step_list: List[str]
    current_step_index: int
    step_results: List[str]
    final_report: str
    clarification_question: str
    is_blocked: bool

# --- NODE A: Planner ---
def planner(state: State):
    print("---PLANNING---")
    prompt = f"Break down this goal into a list of 3 sequential steps. Return ONLY a JSON list of strings: {state['user_goal']}"
    response = llm.invoke(prompt)
    try:
        # Simple parsing for demonstration
        steps = json.loads(response.content)
    except:
        steps = ["Search for info", "Process info", "Format report"]

    return {"step_list": steps, "current_step_index": 0, "step_results": [], "is_blocked": False}

# --- NODE B: Executor ---
def executor(state: State):
    idx = state['current_step_index']
    step = state['step_list'][idx]
    print(f"---EXECUTING STEP {idx+1}: {step}---")

    # Simulate execution
    result = llm.invoke(f"Perform this task and give a brief result: {step}").content
    return {"step_results": state['step_results'] + [result]}

# --- NODE C: Progress Evaluator ---
def progress_evaluator(state: State):
    print("---EVALUATING PROGRESS---")
    # Logic to check for blocking (simulated: block if 'error' in results)
    if "error" in state['step_results'][-1].lower():
        return {"is_blocked": True, "clarification_question": "I encountered an error. How should I proceed?"}

    return {"is_blocked": False}

def router(state: State) -> Literal["executor", "summarizer", "clarification"]:
    if state.get("is_blocked"):
        return "clarification"

    if state['current_step_index'] < len(state['step_list']) - 1:
        state['current_step_index'] += 1
        return "executor"
    else:
        return "summarizer"

# --- OTHER NODES ---
def summarizer(state: State):
    print("---SUMMARIZING---")
    all_results = "\n".join(state['step_results'])
    summary = llm.invoke(f"Summarize these results into a final report: {all_results}").content
    return {"final_report": summary}

def clarification(state: State):
    print("---CLARIFICATION REQUIRED---")
    return state # Terminal node for this flow


## Build graph

In [ ]:
workflow = StateGraph(State)

workflow.add_node("planner", planner)
workflow.add_node("executor", executor)
workflow.add_node("evaluator", progress_evaluator)
workflow.add_node("summarizer", summarizer)
workflow.add_node("clarification", clarification)

workflow.add_edge(START, "planner")
workflow.add_edge("planner", "executor")
workflow.add_edge("executor", "evaluator")

workflow.add_conditional_edges(
    "evaluator",
    router,
    {
        "executor": "executor",
        "summarizer": "summarizer",
        "clarification": "clarification"
    }
)

workflow.add_edge("summarizer", END)
workflow.add_edge("clarification", END)

app = workflow.compile()


In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# Run Example
inputs = {"user_goal": "Research the benefits of LangGraph and write a summary."}
for output in app.stream(inputs):
    print(output)